# SDMAD SST / LST Pipeline — Philippine Area of Responsibility

Temperature **and anomaly** maps over the Philippine Area of Responsibility (PAR) for:

| Product | Source | Record used for the climatology |
|---|---|---|
| **SST** | JAXA GCOM-C/SGLI L3 SST, Google Earth Engine (`ee-chriscandido93`) | 2018 → latest |
| **LST** | MODIS MOD11A2 (Terra) + MYD11A2 (Aqua) 8-day 1 km, Microsoft Planetary Computer `modis-11A2-061` | 2000 → latest |

For LST with both platforms, each platform's anomaly is computed against its own climatology and then averaged
(Terra and Aqua pass at different local times). The first LST run of a window reads that window in every year of
the record; the per-year results are cached in `SDMAD_Products/cache/`, so reruns are fast.

**Before running:** upload this repository to Google Drive as `MyDrive/SDMAD_Products`
(so that `MyDrive/SDMAD_Products/pipeline/` exists). Then run the cells top to bottom.

Outputs are written to `MyDrive/SDMAD_Products/outputs/`:
```
outputs/SST/temperature/   outputs/SST/anomaly/
outputs/LST/temperature/   outputs/LST/anomaly/
outputs/<product>/timeseries/
```

In [ ]:
#@title 1. Mount Google Drive and load the pipeline
PIPELINE_DIR = "/content/drive/MyDrive/SDMAD_Products"  #@param {type:"string"}

import os, sys
from google.colab import drive

drive.mount("/content/drive")
if not os.path.isdir(os.path.join(PIPELINE_DIR, "pipeline")):
    raise FileNotFoundError(
        f"No 'pipeline' folder in {PIPELINE_DIR}. Upload the repository contents to that folder first.")

# numpy is pinned below 2.3 (requirements.txt) because Colab's preinstalled numba requires it. If you see
# "numba requires numpy<2.3", or numpy was imported before this cell: Runtime > Restart session, then run again.
%pip install -q -r "{PIPELINE_DIR}/requirements.txt" geemap

if PIPELINE_DIR not in sys.path:
    sys.path.insert(0, PIPELINE_DIR)
os.chdir(PIPELINE_DIR)
print("Pipeline directory:", PIPELINE_DIR)

In [ ]:
#@title 2. Inputs
#@markdown ### Products
PRODUCTS = "SST + LST"  #@param ["SST + LST", "SST", "LST"]

#@markdown ### Time selection
TIME_MODE = "As of"  #@param ["As of", "Time range", "Month(s)"]

#@markdown **As of** — composite of the `WINDOW_DAYS` days ending on `AS_OF_DATE`
#@markdown (tick `USE_LATEST` to end on the newest available image instead).
USE_LATEST = True  #@param {type:"boolean"}
AS_OF_DATE = "2026-10-15"  #@param {type:"date"}
WINDOW_DAYS = 30  #@param {type:"integer"}

#@markdown **Time range** — inclusive start and end dates (up to 1 year).
START_DATE = "2026-09-01"  #@param {type:"date"}
END_DATE = "2026-09-30"  #@param {type:"date"}

#@markdown **Month(s)** — one map per calendar month from `FIRST_MONTH` to `LAST_MONTH` (YYYY-MM).
FIRST_MONTH = "2026-09"  #@param {type:"string"}
LAST_MONTH = "2026-09"  #@param {type:"string"}

#@markdown ### SST options (GCOM-C, Earth Engine)
SST_PASS = "both"  #@param ["both", "D", "N"]
#@markdown ### LST options (MODIS, Planetary Computer)
LST_PASS = "D"  #@param ["D", "N"]
LST_PLATFORM = "both"  #@param ["both", "terra", "aqua"]
#@markdown Optional Planetary Computer key (raises rate limits; anonymous access works without it).
PC_SDK_SUBSCRIPTION_KEY = ""  #@param {type:"string"}

#@markdown ### Common options
#@markdown Climatology baseline years — leave both at 0 to use the entire GCOM-C record (recommended).
BASELINE_FIRST_YEAR = 0  #@param {type:"integer"}
BASELINE_LAST_YEAR = 0  #@param {type:"integer"}
#@markdown `SAVE_TO` applies to SST; LST is always written to `SDMAD_Products/outputs`.
SAVE_TO = "SDMAD_Products/outputs"  #@param ["SDMAD_Products/outputs", "GEE export task to Drive"]
GEE_PROJECT = "ee-chriscandido93"  #@param {type:"string"}

import datetime as dt
import functools
import pathlib

from pipeline import config, gee, modis, mpc, periods, plot, runner, timeseries

OUTPUT_DIR = pathlib.Path(PIPELINE_DIR) / "outputs"
mpc.set_cache_dir(pathlib.Path(PIPELINE_DIR) / "cache")
if PC_SDK_SUBSCRIPTION_KEY:
    os.environ["PC_SDK_SUBSCRIPTION_KEY"] = PC_SDK_SUBSCRIPTION_KEY
SELECTED = ["SST", "LST"] if PRODUCTS == "SST + LST" else [PRODUCTS]
PASSES = {"SST": SST_PASS, "LST": LST_PASS}
BASELINE = (BASELINE_FIRST_YEAR, BASELINE_LAST_YEAR) if BASELINE_FIRST_YEAR and BASELINE_LAST_YEAR else None
EXPORT_MODE = "local" if SAVE_TO == "SDMAD_Products/outputs" else "drive"

if TIME_MODE == "As of":
    TARGETS = functools.partial(
        runner.resolve_targets, as_of=None if USE_LATEST else dt.date.fromisoformat(AS_OF_DATE),
        window_days=WINDOW_DAYS)
    print(f"As of {'latest available image' if USE_LATEST else AS_OF_DATE}, {WINDOW_DAYS}-day window")
elif TIME_MODE == "Time range":
    TARGETS = functools.partial(runner.resolve_targets, start=START_DATE, end=END_DATE)
    print("Time range", periods.custom_period(START_DATE, END_DATE).label)
else:
    TARGETS = functools.partial(runner.resolve_targets, months=periods.month_labels(FIRST_MONTH, LAST_MONTH))
    print("Months", ", ".join(periods.month_labels(FIRST_MONTH, LAST_MONTH)))
print("Products:", SELECTED, "| passes:", {p: PASSES[p] for p in SELECTED}, "| LST platform:", LST_PLATFORM,
      "| baseline:", BASELINE or "entire record", "| output:", SAVE_TO)

In [ ]:
#@title 3. Authenticate and initialise Earth Engine (SST and the interactive map)
gee.initialize(GEE_PROJECT)
print("Earth Engine ready, project:", GEE_PROJECT)

In [ ]:
#@title 4. Run the pipeline
results = []
for name in SELECTED:
    results += runner.run_product(
        config.PRODUCTS[name], TARGETS, sat_pass=PASSES[name], baseline=BASELINE,
        platform=LST_PLATFORM, export_mode=EXPORT_MODE, out=OUTPUT_DIR, png=True)

if EXPORT_MODE == "drive":
    print("Exports started. Track them at https://code.earthengine.google.com/tasks")
print(f"Done: {len(results)} output(s).")

## Visualisation

In [ ]:
#@title 5. Output maps
import matplotlib.pyplot as plt

fig = plot.results_panel(results)
if fig is None:
    print("No local GeoTIFFs to show (SST in GEE Drive export mode, or no data). Use the interactive map below.")
else:
    plt.show()

In [ ]:
#@title 6. Interactive map (zoom, toggle layers; the inspector reads SST values)
import ee
import geemap
import ipyleaflet
from google.colab import output

output.enable_custom_widget_manager()

m = geemap.Map(center=[15, 125], zoom=5)
m.add_basemap("Esri.WorldGrayCanvas")
colorbars = {}
for r in results:
    product = config.PRODUCTS[r["product"]]
    if r["kind"] == "anomaly":
        vmin, vmax, _ = config.ANOMALY_VIS
        vis = {"min": vmin, "max": vmax, "palette": config.ANOMALY_PALETTE}
        label = f"{product.name} anomaly (°C)"
    else:
        vmin, vmax = product.temp_vis
        vis = {"min": vmin, "max": vmax, "palette": config.TEMP_PALETTE}
        label = f"{product.name} (°C)"
    layer_name = f"{product.name} {r['kind']} {r['period'].label}"
    if "image" in r:      # SST: Earth Engine image, rendered by Earth Engine
        m.addLayer(r["image"].clip(gee.par_geometry()), vis, layer_name)
    elif "tif" in r:      # LST: local GeoTIFF from the Planetary Computer, drawn as an overlay
        url, bounds = plot.overlay_png(r["tif"], r["kind"], product)
        m.add_layer(ipyleaflet.ImageOverlay(url=url, bounds=bounds, name=layer_name))
    colorbars.setdefault(label, vis)

par_outline = ee.FeatureCollection([ee.Feature(gee.par_geometry())]).style(
    color="000000", fillColor="00000000", width=2)
m.addLayer(par_outline, {}, "PAR boundary")
for label, vis in colorbars.items():
    m.add_colorbar(vis, label=label)
m

In [ ]:
#@title 7. PAR-mean monthly time series over the entire record
TS_PRODUCT = "SST"  #@param ["SST", "LST"]
#@markdown LST reads every 8-day composite since 2000 on a coarse grid: the first run takes a while
#@markdown (finished years are cached in `SDMAD_Products/cache/`, so later runs only read the current year).
import csv

product = config.PRODUCTS[TS_PRODUCT]
if TS_PRODUCT == "SST":
    coll = gee.collection(product, PASSES["SST"])
    first, latest = gee.date_range(coll)
    print(f"Computing monthly PAR means {first} -> {latest} (this can take a minute)...")
    series = timeseries.monthly_area_mean(coll, first, latest)
    rows = timeseries.monthly_anomalies(series, *(BASELINE or (None, None)))
    tag = PASSES["SST"]
else:
    plats = modis.platforms_for(LST_PLATFORM)
    latest = modis.latest_date(plats)
    first = min(modis.RECORD_START[p] for p in plats)
    print(f"Computing monthly PAR-land means {first} -> {latest} for {', '.join(plats)}...")
    per_platform = modis.monthly_series(first, latest, PASSES["LST"], LST_PLATFORM)
    rows = timeseries.combine_platforms(
        {p: timeseries.monthly_anomalies(s, *(BASELINE or (None, None))) for p, s in per_platform.items()})
    tag = f"{PASSES['LST']}_{LST_PLATFORM}"

ts_dir = OUTPUT_DIR / product.name / "timeseries"
ts_dir.mkdir(parents=True, exist_ok=True)
stem = ts_dir / f"{product.name}_PAR_monthly_{tag}"
with open(f"{stem}.csv", "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["month", "mean_degC", "climatology_degC", "anomaly_degC"])
    for d, v, c, a in rows:
        w.writerow([d.strftime("%Y-%m")] + ["" if x is None else round(x, 3) for x in (v, c, a)])

fig = plot.timeseries_figure(rows, product)
fig.savefig(f"{stem}.png", dpi=150)
plt.show()
print("Saved", f"{stem}.csv", "and", f"{stem}.png")